# NB07. XGBoost Teacher Robustness

**목적**: LightGBM 대신 XGBoost를 teacher로 사용해도 C1(R²≠AT4)과 C3(depth-1 동치)이 유지되는지 확인.

**Dependencies**: NB01 (datasets)

**핵심 질문**: 주요 발견이 teacher 아키텍처에 독립적인가?

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, json, time
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, roc_auc_score
from scipy.stats import spearmanr

from decentra._utils import transform_logit_to_score
from decentra.surrogate import TreeSurrogate, EBMSurrogate, LinearSurrogate, BinningSurrogate
from decentra.metrics.attribution import attribution_fidelity

NB01_DIR = Path('../outputs/NB01')
OUT_DIR = Path('../outputs/NB07')
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB01_DIR / 'datasets.pkl', 'rb') as f:
    datasets = pickle.load(f)

RS = 42
print('Ready')

Ready


In [2]:
all_results = {}
SURR_SUBSET = ['Tree-d1', 'EBM', 'Ridge', 'OptBin+Ridge']

for data_name in datasets:
    d = datasets[data_name]
    X_train, X_test = d['X_train'], d['X_test']
    y_train, y_test = d['y_train'], d['y_test']
    feature_names = d['feature_names']
    n_features = len(feature_names)
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=RS)
    
    print(f'\n{"="*70}')
    print(f'  {data_name}: XGBoost Teacher')
    print(f'{"="*70}')
    
    # XGBoost teacher
    t0 = time.time()
    teacher = xgb.XGBClassifier(
        max_depth=6, n_estimators=1000, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=50,
        eval_metric='logloss', verbosity=0, random_state=RS, n_jobs=-1)
    teacher.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    
    prob_te = teacher.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob_te)
    score_te = transform_logit_to_score(prob_te)
    score_tr = transform_logit_to_score(teacher.predict_proba(X_train)[:, 1])
    
    # BB SHAP
    bb_shap = np.array(shap.TreeExplainer(teacher).shap_values(X_test), dtype=np.float32)
    print(f'  XGB AUC={auc:.4f}, SHAP={bb_shap.shape}, {time.time()-t0:.0f}s')
    
    reject = prob_te >= np.percentile(prob_te, 90)
    all_results[data_name] = {'XGB_AUC': round(auc, 4)}
    
    # Subset surrogates
    surr_map = {
        'Tree-d1': TreeSurrogate(max_depth=1, verbose=0),
        'EBM': EBMSurrogate(interactions=0, n_jobs=4),
        'Ridge': LinearSurrogate(method='ridgecv'),
        'OptBin+Ridge': BinningSurrogate(max_n_bins=10),
    }
    
    y_tr_score = score_tr[X_train.index.get_indexer(X_tr.index)]
    y_val_score = score_tr[X_train.index.get_indexer(X_val.index)]
    
    for surr_name, surr in surr_map.items():
        t0 = time.time()
        if isinstance(surr, TreeSurrogate):
            surr.fit(X_tr, y_tr_score, eval_set=(X_val, y_val_score))
        elif isinstance(surr, BinningSurrogate):
            surr.fit(X=X_train, y_logit=score_tr, feature_names=feature_names)
        else:
            surr.fit(X=X_train, y_logit=score_tr)
        
        surr_pred = surr.predict(X_test)
        surr_contribs = surr.contributions(X_test)
        
        r2 = round(r2_score(score_te, surr_pred), 4)
        af = attribution_fidelity(bb_shap, surr_contribs, reject, bb_sign=1, surr_sign=-1)
        
        all_results[data_name][surr_name] = {
            'R2': r2, **{k: round(v, 4) for k, v in af.items()}}
        
        print(f'  {surr_name:16s}  R2={r2:.3f}  AT1={af["AdvTop1"]:.3f}  AT4={af["AdvTop4"]:.3f}  ({time.time()-t0:.0f}s)')

with open(OUT_DIR / 'xgb_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved to {OUT_DIR}/')


  GMSC: XGBoost Teacher


  XGB AUC=0.8662, SHAP=(30000, 10), 2s


  Tree-d1           R2=0.952  AT1=0.790  AT4=0.913  (3s)


  EBM               R2=0.956  AT1=0.822  AT4=0.916  (40s)


  Ridge             R2=0.330  AT1=0.318  AT4=0.554  (1s)


...

  XGB AUC=0.7485, SHAP=(61503, 41), 16s


  Tree-d1           R2=0.901  AT1=0.934  AT4=0.782  (6s)


  EBM               R2=0.934  AT1=0.920  AT4=0.861  (432s)


  Ridge             R2=0.818  AT1=0.000  AT4=0.615  (3s)


  OptBin+Ridge      R2=0.858  AT1=0.913  AT4=0.582  (7s)

Saved to ..\outputs\NB07/
